# Pepper Root Deep Dive
Round 1 analysis — focus on INTARIAN_PEPPER_ROOT (IPR).

Goal: determine whether v8's buy-and-hold-40 thesis is empirically supported,
and hunt for exploitable bot signals ("Olivia signal").

In [3]:
# Cell 1: Load all 3 days of price + trade data, sanity-check
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = '../r1_data_capsule'
DAYS = [-2, -1, 0]

# --- Load prices ---
prices_frames = []
for d in DAYS:
    df = pd.read_csv(f'{DATA_DIR}/prices_round_1_day_{d}.csv', sep=';')
    prices_frames.append(df)
prices = pd.concat(prices_frames, ignore_index=True)

# --- Load trades ---
trades_frames = []
for d in DAYS:
    df = pd.read_csv(f'{DATA_DIR}/trades_round_1_day_{d}.csv', sep=';')
    df['day'] = d  # trades file doesn't have a day column
    trades_frames.append(df)
trades = pd.concat(trades_frames, ignore_index=True)

# --- Filter to pepper root only ---
ipr_prices = prices[prices['product'] == 'INTARIAN_PEPPER_ROOT'].copy()
ipr_trades = trades[trades['symbol'] == 'INTARIAN_PEPPER_ROOT'].copy()

# --- Sanity checks ---
print('=== PRICES ===')
print(f'Total rows: {len(prices):,}')
print(f'Products: {prices["product"].unique()}')
print(f'IPR rows: {len(ipr_prices):,}')
print(f'\nTimesteps per day (IPR):')
for d in DAYS:
    n = len(ipr_prices[ipr_prices['day'] == d])
    ts_range = ipr_prices[ipr_prices['day'] == d]['timestamp']
    print(f'  Day {d:+d}: {n:,} rows, timestamp {ts_range.min()} → {ts_range.max()}')

print(f'\nColumn dtypes:\n{ipr_prices.dtypes}')
print(f'\nNull counts (IPR prices):\n{ipr_prices.isnull().sum()}')

print('\n=== TRADES ===')
print(f'Total rows: {len(trades):,}')
print(f'IPR trades: {len(ipr_trades):,}')
for d in DAYS:
    n = len(ipr_trades[ipr_trades['day'] == d])
    print(f'  Day {d:+d}: {n:,} trades')
print(f'\nColumn dtypes:\n{ipr_trades.dtypes}')
print(f'\nSample IPR prices (first 3 rows):')
ipr_prices.head(3)

=== PRICES ===
Total rows: 60,000
Products: ['INTARIAN_PEPPER_ROOT' 'ASH_COATED_OSMIUM']
IPR rows: 30,000

Timesteps per day (IPR):
  Day -2: 10,000 rows, timestamp 0 → 999900
  Day -1: 10,000 rows, timestamp 0 → 999900
  Day +0: 10,000 rows, timestamp 0 → 999900

Column dtypes:
day                  int64
timestamp            int64
product             object
bid_price_1        float64
bid_volume_1       float64
bid_price_2        float64
bid_volume_2       float64
bid_price_3        float64
bid_volume_3       float64
ask_price_1        float64
ask_volume_1       float64
ask_price_2        float64
ask_volume_2       float64
ask_price_3        float64
ask_volume_3       float64
mid_price          float64
profit_and_loss    float64
dtype: object

Null counts (IPR prices):
day                    0
timestamp              0
product                0
bid_price_1         1216
bid_volume_1        1216
bid_price_2        10587
bid_volume_2       10587
bid_price_3        29555
bid_volume_3       2

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-2,0,INTARIAN_PEPPER_ROOT,9992.0,17.0,NaN,NaN,NaN,NaN,10005.0,9.0,10008.0,17.0,NaN,NaN,9998.5,0.0
2,-2,100,INTARIAN_PEPPER_ROOT,9995.0,11.0,9992.0,16.0,NaN,NaN,10006.0,11.0,10008.0,16.0,NaN,NaN,10000.5,0.0
4,-2,200,INTARIAN_PEPPER_ROOT,9995.0,12.0,NaN,NaN,NaN,NaN,10008.0,20.0,NaN,NaN,NaN,NaN,10001.5,0.0


In [ ]:
# Cell 2: Mid-price plot + daily drift
#
# "Drift" = last mid-price minus first mid-price for that day.
# If v8's thesis ("IPR drifts up") is correct, drift should be
# consistently positive across all 3 days.

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

drift_results = []
for i, d in enumerate(DAYS):
    day_data = ipr_prices[ipr_prices['day'] == d].sort_values('timestamp')
    ts = day_data['timestamp'].values
    mid = day_data['mid_price'].values

    first_mid = mid[0]
    last_mid = mid[-1]
    drift = last_mid - first_mid
    drift_results.append({'day': d, 'first_mid': first_mid, 'last_mid': last_mid,
                          'drift': drift, 'min_mid': mid.min(), 'max_mid': mid.max()})

    ax = axes[i]
    ax.plot(ts, mid, linewidth=0.5, color='steelblue')
    ax.set_title(f'Day {d:+d} — drift = {drift:+.1f}')
    ax.set_xlabel('timestamp')
    ax.set_ylabel('mid price')
    ax.axhline(first_mid, color='gray', linestyle='--', alpha=0.5, label='open')
    ax.axhline(last_mid, color='red', linestyle='--', alpha=0.5, label='close')
    ax.legend(fontsize=8)

plt.suptitle('INTARIAN_PEPPER_ROOT — Mid Price by Day', fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
drift_df = pd.DataFrame(drift_results)
print('\n=== Daily Drift Summary ===')
print(drift_df.to_string(index=False))

mean_drift = drift_df['drift'].mean()
std_drift = drift_df['drift'].std()
print(f'\nMean drift: {mean_drift:+.1f}')
print(f'Std drift:  {std_drift:.1f}')
print(f'\nBuy-40-at-open PnL estimate per day (drift × 40):')
for _, row in drift_df.iterrows():
    pnl = row['drift'] * 40
    print(f'  Day {row["day"]:+.0f}: {pnl:+,.0f}')
print(f'  Mean: {mean_drift * 40:+,.0f}')

In [ ]:
# Cell 3: Trade size histogram + overlay on price chart
#
# "Olivia signal" hypothesis: some bot trades fixed sizes at
# predictable points (e.g., daily highs/lows). If we can identify
# which size belongs to which bot, we can front-run or fade them.

# --- 3a: Trade size histogram ---
fig, ax = plt.subplots(figsize=(12, 4))
size_counts = ipr_trades['quantity'].value_counts().sort_index()
ax.bar(size_counts.index, size_counts.values, color='steelblue', edgecolor='black')
ax.set_xlabel('Trade Size (quantity)')
ax.set_ylabel('Frequency')
ax.set_title('IPR Trade Size Histogram (all 3 days)')
for sz, cnt in size_counts.items():
    ax.annotate(f'{cnt}', (sz, cnt), textcoords='offset points',
                xytext=(0, 5), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print('\n=== Trade size frequency ===')
print(size_counts.to_string())

# Identify "repeating" sizes: appear at least 10 times total
repeating_sizes = size_counts[size_counts >= 10].index.tolist()
print(f'\nRepeating sizes (>=10 occurrences): {repeating_sizes}')

In [ ]:
# Cell 3b: Overlay trades of each repeating size on the price chart
#
# For each repeating size, we scatter those trades on top of the
# mid-price time series. If a size clusters at highs or lows,
# that's a strong signal of a predictable bot.

colors = plt.cm.tab10.colors

fig, axes = plt.subplots(len(DAYS), 1, figsize=(18, 5 * len(DAYS)))
if len(DAYS) == 1:
    axes = [axes]

for i, d in enumerate(DAYS):
    ax = axes[i]
    day_prices = ipr_prices[ipr_prices['day'] == d].sort_values('timestamp')
    day_trades = ipr_trades[ipr_trades['day'] == d]

    # Plot mid price as background
    ax.plot(day_prices['timestamp'], day_prices['mid_price'],
            color='lightgray', linewidth=0.5, label='mid price')

    # Overlay each repeating size
    for j, sz in enumerate(repeating_sizes):
        sz_trades = day_trades[day_trades['quantity'] == sz]
        if len(sz_trades) == 0:
            continue
        ax.scatter(sz_trades['timestamp'], sz_trades['price'],
                   color=colors[j % len(colors)], s=20, alpha=0.7,
                   label=f'size={sz} (n={len(sz_trades)})', zorder=3)

    ax.set_title(f'Day {d:+d} — IPR Trades by Size')
    ax.set_xlabel('timestamp')
    ax.set_ylabel('price')
    ax.legend(fontsize=7, loc='upper left', ncol=2)

plt.tight_layout()
plt.show()